In [3]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class SketchDataset(Dataset):
            
    def __init__(self, images_path, labels_path, indices_path, split):

        self.images = np.load(images_path, mmap_mode="r")
        self.labels = np.load(labels_path, mmap_mode="r")
        splits = np.load(indices_path)
        self.indices = splits[f"{split}_idx"]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):

        actual_index = self.indices[index]
        image = self.images[actual_index].reshape(28, 28)
        label = self.labels[actual_index]

        image = torch.tensor(
            image.copy(),
            dtype = torch.float32
        ) / 255.0
        
        image = image.unsqueeze(0)
        label = torch.tensor(label, dtype = torch.long)
        
        return image, label
        
        

In [4]:
# instantiate 3 datasets

DATA_DIR = "../data/processed"

train_dataset = SketchDataset(
    f"{DATA_DIR}/images.npy",
    f"{DATA_DIR}/labels.npy",
    f"{DATA_DIR}/split_indices.npz",
    "train"
)

val_dataset = SketchDataset(
    f"{DATA_DIR}/images.npy",
    f"{DATA_DIR}/labels.npy",
    f"{DATA_DIR}/split_indices.npz",
    "val"
)

test_dataset = SketchDataset(
    f"{DATA_DIR}/images.npy",
    f"{DATA_DIR}/labels.npy",
    f"{DATA_DIR}/split_indices.npz",
    "test"
)
    

In [5]:
image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Image dtype:", image.dtype)
print("Image range:", image.min().item(), image.max().item())
print("Label:", label.item())

Image shape: torch.Size([1, 28, 28])
Image dtype: torch.float32
Image range: 0.0 1.0
Label: 3


In [6]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


In [7]:
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

Batch image shape: torch.Size([64, 1, 28, 28])
Batch label shape: torch.Size([64])
Image dtype: torch.float32
Label dtype: torch.int64
